In [1]:
import torch
import os

import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, quantize_static, CalibrationDataReader, QuantType

import numpy as np

from OcclusionDetection.OcclusionDetectionCNN import OcclusionDetectorCNN, TinyOcclusionCNN

/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Create Onnx

In [2]:
model = TinyOcclusionCNN()
model.load_state_dict(torch.load(os.path.join(os.environ["WEIGHTS"], "occlusion.pth")))

<All keys matched successfully>

In [3]:
dummy_input = torch.randn(1,3,240,240, dtype=torch.torch.float32)
torch.onnx.export(model, dummy_input, "occupancy.onnx", opset_version=13)

# Optimize Onnx (Dynamic Quantization)

In [5]:
model_onnx = onnx.load("piece_cnn.onnx")
onnx.checker.check_model(model_onnx)

In [6]:
model_fp32 = "piece_cnn.onnx"
model_int8 = "piece_cnn_quantized_dynamic.onnx"

quantize_dynamic(
    model_input=model_fp32,
    model_output=model_int8,
    weight_type=QuantType.QUInt8,  # or QuantType.QUInt8
    # op_types_to_quantize=["MatMul", "Gemm"]
)

# Load and Test Onnx

In [ ]:
print(ort.get_available_providers())
session = ort.InferenceSession(
    "occlusion.onnx",
    providers=["CPUExecutionProvider"]
    # providers=["CUDAExecutionProvider"]

    # "piece_cnn_quantized_dynamic.onnx",
    # "piece_cnn_quantized_static.onnx",
    # providers=["OpenVINOExecutionProvider"]
)
print(session.get_providers())

input_name = session.get_inputs()[0].name
output_names = [session.get_outputs()[i].name for i in range(len(session.get_outputs()))]

input_name, output_names

['OpenVINOExecutionProvider', 'CPUExecutionProvider']
['CPUExecutionProvider']


('input.1', ['419'])

In [6]:
x = np.random.randn(1,3,240,240).astype(np.float32)

In [7]:
%%timeit
outputs = session.run(output_names, {input_name: x})

3.79 ms ± 109 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [13]:
outputs = session.run(output_names, {input_name: x})
outputs[0].shape, outputs[0][0,0]

((1, 1), 0.59197515)

# Optimize Onnx (Static Quantization)

In [11]:
from Dataset.DataSetLoaders import ChessDataset
from PieceDetection.PieceCropper_3D import PieceCropper

In [12]:
ds = ChessDataset.ChessDataset(
    config={
        "img_size": (640,640)
    }
)

def prep(entry):
    img, lbl = entry
    PieceCropper.piece_cropper.set_img(img, lbl["corners"])
    board_split = PieceCropper.piece_cropper.process_img()
    return board_split.unsqueeze(0)


class CalibrationDataReader(CalibrationDataReader):
    def __init__(self, input_name):
        self.input_name = input_name
        self.data_iter = iter([
            {input_name: prep(ds[_]).numpy()}
            for _ in range(10)
        ])

    def get_next(self):
        return next(self.data_iter, None)

In [13]:
# load model to find input name
model_fp32 = "piece_cnn.onnx"
session = ort.InferenceSession(model_fp32, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name

dr = CalibrationDataReader(input_name)

quantize_static(
    model_input=model_fp32,
    model_output="piece_cnn_quantized_static.onnx",
    calibration_data_reader=dr,
    weight_type=QuantType.QInt8,
)
print("✅ Static quantization done")

✅ Static quantization done
